# C'est le notebook correspondant mais je ne garantis pas sa maintenance ( le gros block est un copié coller de main(cfg))

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
import torch

In [2]:
from src.logger.hydra import init_config



In [3]:
from main import * 

In [7]:
cfg=init_config(overrides=[])

In [8]:
BATCH_SIZE = 128
num_workers=2
TRAIN_IMAGES_PATH = 'train.h5'
VAL_IMAGES_PATH = 'val.h5'
TEST_IMAGES_PATH = 'test.h5'
SEED = 0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
name_out_submit = (
    "submit/" + cfg.name_out_submit + str(random.randint(0, 255)) + ".csv"
)
#! Preprocessing 
print("Preprocessing the dataset",flush=True)

preprocessing = get_transform(transform_name=cfg.transform_name)
train_dataset = BaselineDataset(TRAIN_IMAGES_PATH, preprocessing, 'train')
val_dataset = BaselineDataset(VAL_IMAGES_PATH, preprocessing, 'train')
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE,num_workers=num_workers)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE,num_workers=num_workers)   
#* Apply the preprocesing to the dataset

print("loading the model")
Main_model= baseLine(device)

feature_extractor=Main_model.feature_extractor


# --- Setup functions
OPTIMIZER = cfg.optimizer.optimizer
OPTIMIZER_PARAMS = cfg.optimizer.optimizer_params #{'lr': 0.001}
print("optimizer params",OPTIMIZER_PARAMS,type(OPTIMIZER_PARAMS))
LOSS = cfg.loss
METRIC = cfg.metric
NUM_EPOCHS = cfg.num_epochs
PATIENCE = cfg.patience
linear_probing=Main_model.linear_probing
# Load function 
metric = getattr(torchmetrics, METRIC)('binary')

optimizer = getattr(torch.optim, OPTIMIZER)(linear_probing.parameters(), **OPTIMIZER_PARAMS)
criterion = getattr(torch.nn, LOSS)()
print("the optimizer",optimizer)

#* ---precompute the features---
print("Precompute the features",flush=True)
x,y=precompute(train_dataloader, feature_extractor, device)
print("We did it once",flush=True)
train_dataset = PrecomputedDataset(features=x, labels=y)
x,y=precompute(val_dataloader, feature_extractor, device)
val_dataset = PrecomputedDataset(features=x, labels=y)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE,num_workers=num_workers)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE,num_workers=num_workers)

#* ---train the model---
print("Start training the last layer",flush=True)

linear_probing = train(linear_probing,NUM_EPOCHS,train_dataloader,val_dataloader,optimizer,criterion,metric,PATIENCE,device)

#* Test the model
test_dataset = BaselineDataset(TEST_IMAGES_PATH, preprocessing, 'test')
print("Test the model",flush=True)
test_model(model=Main_model,test_dataset=test_dataset,device=device,BATCH_SIZE=BATCH_SIZE,test_ids=test_dataset.image_ids,name_out=name_out_submit)


Preprocessing the dataset


loading the model


Using cache found in /raid/home/bournez_pie/.cache/torch/hub/facebookresearch_dinov2_main


optimizer params {'lr': 0.001} <class 'omegaconf.dictconfig.DictConfig'>
the optimizer Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Precompute the features


  0%|          | 0/782 [00:00<?, ?it/s]/raid/home/bournez_pie/mva_geom/mva_geom_24/DLMI/project/venv/lib/python3.8/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
/raid/home/bournez_pie/mva_geom/mva_geom_24/DLMI/project/venv/lib/python3.8/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of al

KeyboardInterrupt: 